In [0]:
def test_rolling_avg_balance(spark):
    from pyspark.sql.window import Window
    from pyspark.sql.functions import col, avg, to_date

    data = [
        ("A1", "2024-01-01", 1000),
        ("A1", "2024-01-02", 2000),
        ("A1", "2024-01-03", 3000)
    ]

    columns = ["account_id", "date", "balance"]
    df = spark.createDataFrame(data, columns) \
              .withColumn("date", to_date("date"))

    window_spec = Window.partitionBy("account_id") \
                        .orderBy("date") \
                        .rowsBetween(-6, 0)

    result_df = df.withColumn(
        "rolling_avg_balance",
        avg("balance").over(window_spec)
    )

    result = [r["rolling_avg_balance"] for r in result_df.collect()]

    try:
        assert round(result[2], 2) == 2000.00
        print("✅ Test Passed")
    except AssertionError:
        print("❌ Test Failed")

test_rolling_avg_balance(spark)